In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, sys
os.chdir("/content/drive/MyDrive/Cape Stone Project 2026")
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())


In [3]:
# Phase 8: Feature Engineering — RFM + Support Features + Churn Flag

import pandas as pd
import os

# -----------------------------
# Load cleaned data
# -----------------------------
customers = pd.read_csv("/content/drive/MyDrive/Cape Stone Project 2026/data/processed/customers_clean.csv")
orders = pd.read_csv("/content/drive/MyDrive/Cape Stone Project 2026/data/processed/orders_clean.csv")
support_tickets = pd.read_csv("/content/drive/MyDrive/Cape Stone Project 2026/data/processed/support_tickets_clean.csv")

# -----------------------------
# Compute churn flag (from Phase 7 logic)
# -----------------------------
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
cutoff = pd.Timestamp("2026-09-24") - pd.Timedelta(days=365)

last_order = orders.groupby("customer_id")["order_date"].max().reset_index()
last_order["churn_flag"] = (last_order["order_date"] < cutoff).astype(int)

# -----------------------------
# RFM Features
# -----------------------------
recency = orders.groupby("customer_id")["order_date"].max().apply(
    lambda x: (pd.Timestamp("2026-09-24") - x).days
)
frequency = orders[orders["order_status"]=="Completed"].groupby("customer_id")["order_id"].count()
monetary = orders[orders["order_status"]=="Completed"].groupby("customer_id")["total_amount"].sum()

rfm = pd.DataFrame({
    "customer_id": recency.index,
    "recency": recency.values,
    "frequency": frequency.reindex(recency.index, fill_value=0).values,
    "monetary": monetary.reindex(recency.index, fill_value=0).values
})

# -----------------------------
# Support Ticket Features
# -----------------------------
ticket_summary = support_tickets.groupby("customer_id").agg({
    "ticket_id":"count",
    "satisfaction_score":"mean",
    "resolution_time_hours":"mean"
}).rename(columns={
    "ticket_id":"ticket_count",
    "satisfaction_score":"avg_satisfaction",
    "resolution_time_hours":"avg_resolution_time"
}).reset_index()

# -----------------------------
# Merge All Features
# -----------------------------
features = customers.merge(rfm, on="customer_id", how="left")
features = features.merge(ticket_summary, on="customer_id", how="left")
features = features.merge(last_order[["customer_id", "churn_flag"]], on="customer_id", how="left")

features = features.fillna(0)

# -----------------------------
# Save Features Robustly
# -----------------------------
processed_dir = "/content/drive/MyDrive/Cape Stone Project 2026/data/processed"
os.makedirs(processed_dir, exist_ok=True)

features.to_csv(os.path.join(processed_dir, "features.csv"), index=False)

print("✅ Features dataset with churn_flag saved at:", os.path.join(processed_dir, "features.csv"))
print("Shape:", features.shape)
print("Columns:", features.columns.tolist())

# -----------------------------
# Quick Validation (numeric only)
# -----------------------------
numeric_features = features.select_dtypes(include=["number"])
corr = numeric_features.corr()["churn_flag"].sort_values(ascending=False)

print("\nCorrelation with churn_flag (numeric features only):")
print(corr)


✅ Features dataset with churn_flag saved at: /content/drive/MyDrive/Cape Stone Project 2026/data/processed/features.csv
Shape: (5922, 21)
Columns: ['customer_id', 'first_name', 'last_name', 'email', 'phone', 'gender', 'date_of_birth', 'signup_date', 'state', 'city', 'membership_tier', 'preferred_channel', 'marketing_opt_in', 'referral_source', 'recency', 'frequency', 'monetary', 'ticket_count', 'avg_satisfaction', 'avg_resolution_time', 'churn_flag']

Correlation with churn_flag (numeric features only):
churn_flag             1.000000
recency                0.801090
frequency              0.026021
ticket_count           0.020951
monetary               0.015649
avg_satisfaction       0.007918
avg_resolution_time   -0.000169
Name: churn_flag, dtype: float64
